# 02 — Demo Retrieval (BM25 vs IndoSBERT vs Hybrid RRF)

**Notebook andalan untuk presentasi.** Ketik sebuah query, lalu bandingkan hasil ketiga pendekatan secara berdampingan.

**Prasyarat:** sudah menjalankan `scripts/00_download_model.py`, `01_extract.py`, `02_chunk.py`, dan `03_build_index.py`.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src.config import load_config, resolve_path
from src.bm25_retriever import BM25Retriever
from src.semantic_retriever import SemanticRetriever
from src.fusion import reciprocal_rank_fusion

cfg = load_config()
index_dir = resolve_path(cfg['paths']['index_dir'])
processed = resolve_path(cfg['paths']['processed_dir'])

# Peta id -> teks untuk menampilkan isi pasal
id2text = {json.loads(l)['id']: json.loads(l)['text']
           for l in open(processed / 'chunks.jsonl', encoding='utf-8')}

bm25 = BM25Retriever.load(index_dir / 'bm25.pkl')
sem = SemanticRetriever.load(index_dir / 'faiss')
print('Index siap. Total dokumen:', len(bm25.doc_ids))

Index siap. Total dokumen: 859


In [2]:
TOP_K = cfg['retrieval']['top_k']
RRF_K = cfg['fusion']['k']

def bandingkan(query, top_k=5):
    """Tampilkan top-k hasil dari BM25, IndoSBERT, dan Hybrid berdampingan."""
    bm = bm25.search(query, top_k=TOP_K)
    se = sem.search(query, top_k=TOP_K)
    hy = reciprocal_rank_fusion([bm, se], k=RRF_K, top_k=TOP_K)
    out = pd.DataFrame({
        'BM25': [d for d, _ in bm[:top_k]],
        'IndoSBERT': [d for d, _ in se[:top_k]],
        'Hybrid (RRF)': [d for d, _ in hy[:top_k]],
    })
    out.index = [f'#{i+1}' for i in range(len(out))]
    return out

## Demo 1 — Query bahasa sehari-hari
Query memakai kata yang **berbeda** dari teks UU. Di sinilah IndoSBERT unggul.

In [3]:
bandingkan('berapa pesangon kalau dipecat?')

c:\Users\satri\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 1/1 [00:00<00:00,  9.51it/s]


,BM25,IndoSBERT,Hybrid (RRF)
#1,KETENAGAKERJAAN_PASAL_167,KETENAGAKERJAAN_PASAL_165,KETENAGAKERJAAN_PASAL_165
#2,KETENAGAKERJAAN_PASAL_162_dup1,KETENAGAKERJAAN_PASAL_169,KETENAGAKERJAAN_PASAL_162_dup1
#3,KETENAGAKERJAAN_PASAL_172,KETENAGAKERJAAN_PASAL_156_p1,KETENAGAKERJAAN_PASAL_172
#4,KETENAGAKERJAAN_PASAL_165,KETENAGAKERJAAN_PASAL_95,KETENAGAKERJAAN_PASAL_156_p1
#5,KETENAGAKERJAAN_PASAL_166,KETENAGAKERJAAN_PASAL_162_dup1,KETENAGAKERJAAN_PASAL_166


## Demo 2 — Query dengan istilah hukum spesifik
Di sini BM25 (pencocokan kata eksak) biasanya kuat.

In [4]:
bandingkan('bukti elektronik di pengadilan')

Batches: 100%|██████████| 1/1 [00:00<00:00, 25.36it/s]


,BM25,IndoSBERT,Hybrid (RRF)
#1,ITE_PASAL_44,ITE_PASAL_44,ITE_PASAL_44
#2,ITE_PASAL_5,ITE_PASAL_43_p1,ITE_PASAL_5
#3,KONSUMEN_PASAL_45,ITE_PASAL_5,ITE_PASAL_43_p1
#4,ITE_PASAL_18,ITE_PASAL_1_p1,KONSUMEN_PASAL_45
#5,ITE_PASAL_13,ITE_PASAL_43,ITE_PASAL_18


## Lihat isi pasal teratas (untuk verifikasi relevansi)

In [5]:
query = 'barang yang dibeli tidak sesuai pesanan'
hasil = reciprocal_rank_fusion(
    [bm25.search(query, TOP_K), sem.search(query, TOP_K)], k=RRF_K, top_k=3)
for rank, (doc_id, score) in enumerate(hasil, 1):
    print(f'#{rank}  {doc_id}  (RRF={score:.4f})')
    print('   ', id2text[doc_id][:220], '\n')

Batches: 100%|██████████| 1/1 [00:00<00:00, 24.75it/s]

#1  KONSUMEN_PASAL_16  (RRF=0.0328)
    Pasal 16 Pelaku usaha dalam menawarkan barang dan/atau jasa melalui pesanan dilarang untuk : a. tidak menepati pesanan dan/atau kesepakatan waktu penyelesaian sesuai dengan yang dijanjikan; b. tidak menepati janji atas s 

#2  KONSUMEN_PASAL_27  (RRF=0.0320)
    Pasal 27 Pelaku usaha yang memproduksi barang dibebaskan dari tanggung jawab atas kerugian yang diderita konsumen, apabila : a. barang tersebut terbukti seharusnya tidak diedarkan atau tidak dimaksudkan untuk diedarkan;  

#3  KONSUMEN_PASAL_11  (RRF=0.0306)
    Pasal 11 Pelaku usaha dalam hal penjualan yang dilakukan melalui cara obral atau lelang, dilarang mengelabui/menyesatkan konsumen dengan : a. menyatakan barang dan/atau jasa tersebut seolah-olah telah memenuhi standar mu 



## Coba query Anda sendiri (interaktif saat presentasi)

In [6]:
bandingkan('hak anak untuk mendapatkan pendidikan')

Batches: 100%|██████████| 1/1 [00:00<00:00, 25.91it/s]


,BM25,IndoSBERT,Hybrid (RRF)
#1,ANAK_PASAL_9,ANAK_PASAL_9,ANAK_PASAL_9
#2,KONSUMEN_PASAL_4,ANAK_PASAL_49,ANAK_PASAL_51
#3,ANAK_PASAL_51,ANAK_PASAL_72_p1,ANAK_PASAL_49
#4,ANAK_PASAL_14_dup1,ANAK_PASAL_6_dup1,ANAK_PASAL_14_dup1
#5,ANAK_PASAL_91A_p1,KETENAGAKERJAAN_PASAL_71_dup1,ANAK_PASAL_54
